In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import timedelta, datetime as dt_datetime  # Добавлено datetime as dt_datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import os
import time  # Для rate limiting, если понадобится

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, KFold

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# --- Finnhub и BERT ---
try:
    import finnhub
except ImportError:
    print("Библиотека finnhub-python не найдена. Пожалуйста, установите ее: pip install finnhub-python")
    # exit()

try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch
except ImportError:
    print("Библиотеки transformers или torch не найдены. Пожалуйста, установите их: pip install transformers torch")
    # exit()
# --- Конец Finnhub и BERT ---


# Пример настройки GPU для TensorFlow (если используется и MAX_MODEL_WORKERS > 1)
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            print(f"Enabled memory growth for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"Error setting memory growth: {e}. Must be set before GPU init.")

from tinkoff.invest import Client, CandleInterval, RequestError
from tinkoff.invest.utils import now, quotation_to_decimal

try:
    import pandas_ta as ta
except ImportError:
    print("Библиотека pandas_ta не найдена. Пожалуйста, установите ее: pip install pandas_ta")
    # exit()

from config import TOKEN, FINNHUB_API_KEY  # FINNHUB_API_KEY добавлен в импорт

print_lock = Lock()

import warnings

warnings.filterwarnings("ignore")


In [ ]:
# --- Параметры конфигурации (общие) ---
TINKOFF_API_TOKEN = TOKEN
INSTRUMENT_FIGI = "BBG004730N88"  # SBER (Пример, для Finnhub нужен будет тикер)
FEATURE_COLUMN = 'close'
Z_SCORE_THRESHOLD = 3
N_CV_SPLITS = 4
MAX_CV_WORKERS = min(N_CV_SPLITS, (os.cpu_count() or 1) * 2)
TEST_SPLIT_SIZE = 0.2
RANDOM_STATE = 42

SMA_PERIODS = [1, 5, 10]
MACD_FAST = 12
MACD_SLOW = 26
MACD_SIGNAL = 9
ATR_PERIOD = 14
BBANDS_PERIOD = 20
BBANDS_STD_DEV = 2
RSI_PERIOD = 14
STOCH_K = 14
STOCH_D = 3
STOCH_SMOOTH_K = 3

# --- НОВАЯ ГЛОБАЛЬНАЯ КОНСТАНТА для управления разрывами на графиках ---
MAX_GAP_MULTIPLIER_FOR_PLOT = 1.0

# --- Параметры для новостей и BERT ---
BERT_SENTIMENT_MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
NEWS_SENTIMENT_COLUMN_NAME = "news_sentiment_score"
BERT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # Для BERT модели

# Глобальные переменные для BERT модели, токенизатора и Finnhub клиента
BERT_TOKENIZER = None
BERT_MODEL = None
FINNHUB_CLIENT = None



Блок 0: Вспомогательные функции и инициализация ресурсов

In [ ]:
def get_timedelta_from_candle_interval(interval: CandleInterval) -> timedelta:
    # ... (без изменений) ...
    if interval == CandleInterval.CANDLE_INTERVAL_1_MIN:
        return timedelta(minutes=1)
    elif interval == CandleInterval.CANDLE_INTERVAL_5_MIN:
        return timedelta(minutes=5)
    elif interval == CandleInterval.CANDLE_INTERVAL_10_MIN:
        return timedelta(minutes=10)
    elif interval == CandleInterval.CANDLE_INTERVAL_15_MIN:
        return timedelta(minutes=15)
    elif interval == CandleInterval.CANDLE_INTERVAL_30_MIN:
        return timedelta(minutes=30)
    elif interval == CandleInterval.CANDLE_INTERVAL_HOUR:
        return timedelta(hours=1)
    elif interval == CandleInterval.CANDLE_INTERVAL_2_HOUR:
        return timedelta(hours=2)
    elif interval == CandleInterval.CANDLE_INTERVAL_4_HOUR:
        return timedelta(hours=4)
    elif interval == CandleInterval.CANDLE_INTERVAL_DAY:
        return timedelta(days=1)
    elif interval == CandleInterval.CANDLE_INTERVAL_WEEK:
        return timedelta(weeks=1)
    elif interval == CandleInterval.CANDLE_INTERVAL_MONTH:
        return timedelta(days=30)  # Приблизительно, для общего случая
    else:
        with print_lock:
            print(f"Предупреждение: Неизвестный интервал свечи {interval}, используется 1 минута по умолчанию.")
        return timedelta(minutes=1)


def generate_plotted_times_and_values(real_timestamps, values, timeframe_timedelta_expected,
                                      max_gap_multiplier=MAX_GAP_MULTIPLIER_FOR_PLOT):
    # ... (без изменений) ...
    if not hasattr(real_timestamps, '__len__') or len(real_timestamps) == 0:
        return np.array([]), np.array([])
    if not hasattr(values, '__len__') or len(values) == 0:
        return np.array([]), np.array([])

    if len(real_timestamps) != len(values):
        with print_lock:
            print(
                f"ERROR in generate_plotted_times_and_values: Length mismatch between timestamps ({len(real_timestamps)}) and values ({len(values)}). Returning empty arrays.")
        return np.array([]), np.array([])

    if isinstance(real_timestamps, (pd.Series, pd.DatetimeIndex)):
        real_timestamps_np = real_timestamps.to_numpy()
    else:
        real_timestamps_np = np.asarray(real_timestamps)

    if isinstance(values, pd.Series):
        values_np = values.to_numpy()
    else:
        values_np = np.asarray(values)

    plotted_ts_list = [pd.Timestamp(real_timestamps_np[0])]

    for i in range(1, len(real_timestamps_np)):
        current_real_ts_pd = pd.Timestamp(real_timestamps_np[i])
        prev_real_ts_pd = pd.Timestamp(real_timestamps_np[i - 1])
        last_plotted_ts_pd = plotted_ts_list[-1]

        actual_diff = current_real_ts_pd - prev_real_ts_pd

        if actual_diff > timeframe_timedelta_expected * max_gap_multiplier:
            plotted_ts_list.append(last_plotted_ts_pd + timeframe_timedelta_expected)
        else:
            plotted_ts_list.append(last_plotted_ts_pd + actual_diff)

    return np.array(plotted_ts_list), values_np


def load_bert_resources(model_name_str):
    global BERT_TOKENIZER, BERT_MODEL
    if BERT_TOKENIZER is None or BERT_MODEL is None:
        with print_lock:
            print(f"Загрузка BERT модели: {model_name_str} на устройство {BERT_DEVICE}...")
        try:
            BERT_TOKENIZER = AutoTokenizer.from_pretrained(model_name_str)
            BERT_MODEL = AutoModelForSequenceClassification.from_pretrained(model_name_str)
            BERT_MODEL.to(BERT_DEVICE)
            BERT_MODEL.eval()  # Перевод модели в режим оценки
            with print_lock:
                print("BERT модель и токенизатор успешно загружены.")
        except Exception as e:
            with print_lock:
                print(f"Ошибка при загрузке BERT модели: {e}")
            BERT_TOKENIZER = None
            BERT_MODEL = None


def initialize_finnhub_client(api_key):
    global FINNHUB_CLIENT
    if FINNHUB_CLIENT is None:
        with print_lock:
            print("Инициализация Finnhub клиента...")
        try:
            FINNHUB_CLIENT = finnhub.Client(api_key=api_key)
            # Проверка работоспособности клиента (опционально)
            profile = FINNHUB_CLIENT.company_profile2(symbol='AAPL')  # Тестовый запрос
            if profile and 'name' in profile:
                with print_lock:
                    print(f"Finnhub клиент успешно инициализирован. (Тест: {profile['name']})")
            else:
                with print_lock:
                    print(f"Finnhub клиент инициализирован, но тестовый запрос не вернул ожидаемые данные: {profile}")
        except Exception as e:
            with print_lock:
                print(f"Ошибка при инициализации Finnhub клиента: {e}")
            FINNHUB_CLIENT = None



Блок 1: Загрузка и первичная обработка данных (включая новости)

In [ ]:
def get_tinkoff_candles(token, figi, days, interval, model_name=""):
    # ... (без изменений) ...
    with print_lock:
        print(f"[{model_name}] Загрузка данных: {days} дней, интервал {interval.name}")

    def fetch_day_data(day_offset):
        with Client(token) as client:
            end_dt = now() - timedelta(days=day_offset)
            start_dt = end_dt - timedelta(days=1)
            try:
                candles = list(client.get_all_candles(
                    figi=figi,
                    from_=start_dt,
                    to=end_dt,
                    interval=interval
                ))
                return candles
            except RequestError as e:
                with print_lock:
                    print(f"[{model_name}] Ошибка при загрузке данных за {start_dt.date()}: {e}")
                return []

    max_fetch_workers = min(10, days if days > 0 else 1)
    all_candles_nested = []
    with ThreadPoolExecutor(max_workers=max_fetch_workers) as executor:
        futures = [executor.submit(fetch_day_data, day_offset) for day_offset in range(days)]
        for future in as_completed(futures):
            all_candles_nested.append(future.result())

    all_candles = [candle for sublist in all_candles_nested for candle in sublist if sublist]

    if not all_candles:
        with print_lock:
            print(f"[{model_name}] Не удалось загрузить свечи.")
        return pd.DataFrame()

    df = pd.DataFrame([{
        'time': pd.Timestamp(c.time).tz_localize(None),  # Уже tz-naive UTC
        'open': float(quotation_to_decimal(c.open)),
        'high': float(quotation_to_decimal(c.high)),
        'low': float(quotation_to_decimal(c.low)),
        'close': float(quotation_to_decimal(c.close)),
        'volume': c.volume
    } for c in all_candles])

    if df.empty:
        return pd.DataFrame()

    df = df.drop_duplicates(subset=['time'])
    df = df.set_index('time').sort_index()
    df.dropna(inplace=True)  # Базовое удаление NaN, если есть

    with print_lock:
        print(f"[{model_name}] Загружено всего {len(df)} уникальных свечей.")
        print(f"[{model_name}] Размер DataFrame после загрузки и базовой очистки: {df.shape}")
    return df


# --- Функции для новостей ---
def fetch_company_news_finnhub(client, symbol, start_date_str, end_date_str, model_name="DefaultModel", max_retries=3,
                               retry_delay_sec=5):
    if client is None:
        with print_lock:
            print(f"[{model_name}] Finnhub клиент не инициализирован. Пропуск загрузки новостей.")
        return []
    with print_lock:
        print(f"[{model_name}] Запрос новостей для {symbol} с {start_date_str} по {end_date_str}...")

    for attempt in range(max_retries):
        try:
            news = client.company_news(symbol, _from=start_date_str, to=end_date_str)
            with print_lock:
                print(f"[{model_name}] Загружено {len(news)} новостей для {symbol}.")
            # Finnhub может вернуть пустой список или список с ошибкой в словаре
            if isinstance(news, list) and all(isinstance(item, dict) for item in news):
                # Фильтрация пустых новостей или новостей без заголовка
                news = [n for n in news if n.get('headline')]
                return news
            else:
                with print_lock:
                    print(
                        f"[{model_name}] Finnhub вернул неожиданный формат новостей: {type(news)}. Попытка {attempt + 1}/{max_retries}")

        except finnhub.FinnhubAPIException as e:  # Обработка специфичных ошибок Finnhub
            with print_lock:
                print(
                    f"[{model_name}] Finnhub API ошибка при загрузке новостей (попытка {attempt + 1}/{max_retries}): {e}")
            if e.status_code == 429:  # Too Many Requests
                with print_lock:
                    print(
                        f"[{model_name}] Достигнут лимит запросов Finnhub. Ожидание {retry_delay_sec * (attempt + 1)} сек...")
                time.sleep(retry_delay_sec * (attempt + 1))  # Экспоненциальная задержка
            elif attempt == max_retries - 1:
                with print_lock:
                    print(f"[{model_name}] Все попытки загрузки новостей для {symbol} не удались.")
                return []  # Возвращаем пустой список при неудаче
        except Exception as e:
            with print_lock:
                print(f"[{model_name}] Общая ошибка при загрузке новостей (попытка {attempt + 1}/{max_retries}): {e}")
            if attempt == max_retries - 1:
                with print_lock:
                    print(f"[{model_name}] Все попытки загрузки новостей для {symbol} не удались.")
                return []
        time.sleep(retry_delay_sec)  # Небольшая задержка между попытками
    return []


def analyze_news_sentiment_bert(news_df, tokenizer, model, text_column='headline', model_name="DefaultModel"):
    if model is None or tokenizer is None:
        with print_lock:
            print(f"[{model_name}] BERT модель/токенизатор не загружены. Анализ Sentiment пропущен.")
        news_df[NEWS_SENTIMENT_COLUMN_NAME] = 0.0
        return news_df

    if news_df.empty or text_column not in news_df.columns:
        with print_lock:
            print(
                f"[{model_name}] DataFrame новостей пуст или отсутствует колонка '{text_column}'. Анализ Sentiment пропущен.")
        if not news_df.empty:
            news_df[NEWS_SENTIMENT_COLUMN_NAME] = 0.0
        return news_df

    texts_to_analyze = news_df[text_column].astype(str).tolist()
    sentiments = []
    with print_lock:
        print(f"[{model_name}] Анализ Sentiment для {len(texts_to_analyze)} новостей с помощью BERT...")

    # Batch processing (если модель поддерживает и есть GPU)
    # Для простоты, здесь итеративно, но для больших объемов лучше батчинг
    for i, text in enumerate(texts_to_analyze):
        if (i + 1) % 50 == 0:  # Логирование прогресса
            with print_lock:
                print(f"[{model_name}] Обработано {i + 1}/{len(texts_to_analyze)} новостей для sentiment...")
        try:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(BERT_DEVICE)
            with torch.no_grad():  # Важно для режима оценки
                logits = model(**inputs).logits

            predicted_class_id = torch.argmax(logits, dim=1).item()
            # Преобразование класса (0-4) в sentiment score (-1.0 до 1.0)
            # 0 -> -1.0 (very negative)
            # 1 -> -0.5 (negative)
            # 2 ->  0.0 (neutral)
            # 3 ->  0.5 (positive)
            # 4 ->  1.0 (very positive)
            sentiment_score = (predicted_class_id - 2) * 0.5
            sentiments.append(sentiment_score)
        except Exception as e:
            with print_lock:
                print(f"[{model_name}] Ошибка анализа sentiment для текста: '{text[:50]}...': {e}")
            sentiments.append(0.0)  # Нейтральный sentiment в случае ошибки

    news_df[NEWS_SENTIMENT_COLUMN_NAME] = sentiments
    with print_lock:
        print(f"[{model_name}] Анализ Sentiment завершен.")
    return news_df


def map_news_to_candles(df_candles, df_news_with_sentiment, aggregation_lookback_timedelta,
                        sentiment_col_name, model_name="DefaultModel",
                        news_time_col='datetime', candle_time_col_is_index=True):
    with print_lock:
        print(f"[{model_name}] Сопоставление новостей со свечами...")

    if df_news_with_sentiment.empty or sentiment_col_name not in df_news_with_sentiment.columns:
        with print_lock:
            print(
                f"[{model_name}] DataFrame новостей пуст или отсутствует колонка sentiment. Возвращаем исходный df_candles.")
        df_candles_out = df_candles.copy()
        df_candles_out[sentiment_col_name] = 0.0
        return df_candles_out

    # Убедимся, что временные колонки в правильном формате
    if not pd.api.types.is_datetime64_any_dtype(df_news_with_sentiment[news_time_col]):
        df_news_with_sentiment[news_time_col] = pd.to_datetime(df_news_with_sentiment[news_time_col], errors='coerce')

    df_news_with_sentiment = df_news_with_sentiment.dropna(subset=[news_time_col])
    df_news_with_sentiment = df_news_with_sentiment.sort_values(by=news_time_col)

    # Подготовка выходного DataFrame
    df_candles_out = df_candles.copy()
    df_candles_out[sentiment_col_name] = 0.0  # Инициализация нулями

    # Итерация по свечам (может быть медленно для больших данных, можно оптимизировать с merge_asof)
    for candle_idx, candle_row in df_candles_out.iterrows():
        candle_time = candle_idx if candle_time_col_is_index else candle_row[
            df_candles.index.name]  # Используем индекс свечи

        # Определение временного окна для новостей
        start_window_time = candle_time - aggregation_lookback_timedelta

        # Фильтрация релевантных новостей
        relevant_news = df_news_with_sentiment[
            (df_news_with_sentiment[news_time_col] <= candle_time) &
            (df_news_with_sentiment[news_time_col] > start_window_time)
            ]

        if not relevant_news.empty:
            # Агрегация (среднее) sentiment score
            aggregated_sentiment = relevant_news[sentiment_col_name].mean()
            df_candles_out.loc[candle_idx, sentiment_col_name] = aggregated_sentiment

    df_candles_out[sentiment_col_name] = df_candles_out[sentiment_col_name].fillna(0.0)
    with print_lock:
        print(f"[{model_name}] Сопоставление новостей завершено. Добавлена колонка '{sentiment_col_name}'.")
        print(f"[{model_name}] Статистика по sentiment: \n{df_candles_out[sentiment_col_name].describe()}")
    return df_candles_out


def add_news_sentiment_feature(df_candles_input: pd.DataFrame,
                               news_processing_config: dict,
                               finnhub_client_global,
                               bert_tokenizer_global,
                               bert_model_global,
                               sentiment_col_name_const: str,
                               model_name: str = "DefaultModel") -> pd.DataFrame:
    if not news_processing_config.get("enabled", False):
        with print_lock:
            print(f"[{model_name}] Обработка новостей отключена в конфигурации.")
        df_out = df_candles_input.copy()
        df_out[sentiment_col_name_const] = 0.0  # Добавляем колонку с нулями, если она ожидается дальше
        return df_out

    if df_candles_input.empty:
        with print_lock:
            print(f"[{model_name}] Входной DataFrame свечей пуст. Пропуск добавления новостной фичи.")
        df_out = df_candles_input.copy()
        if not df_out.empty:  # Если вдруг пустой df, но с колонками
            df_out[sentiment_col_name_const] = 0.0
        return df_out

    symbol_for_news = news_processing_config.get("finnhub_symbol")
    if not symbol_for_news:
        with print_lock:
            print(f"[{model_name}] Тикер Finnhub не указан в конфигурации новостей. Пропуск.")
        df_out = df_candles_input.copy()
        df_out[sentiment_col_name_const] = 0.0
        return df_out

    # Определяем период для запроса новостей
    min_candle_time = df_candles_input.index.min()
    max_candle_time = df_candles_input.index.max()

    # Запрашиваем новости с небольшим запасом назад
    fetch_start_time = min_candle_time - news_processing_config.get("aggregation_lookback_timedelta",
                                                                    pd.Timedelta(hours=0)) \
                       - pd.Timedelta(days=news_processing_config.get("news_fetch_days_offset", 1))

    fetch_start_date_str = fetch_start_time.strftime('%Y-%m-%d')
    fetch_end_date_str = max_candle_time.strftime('%Y-%m-%d')

    # 1. Загрузка новостей
    raw_news_list = fetch_company_news_finnhub(
        client=finnhub_client_global,
        symbol=symbol_for_news,
        start_date_str=fetch_start_date_str,
        end_date_str=fetch_end_date_str,
        model_name=model_name
    )

    if not raw_news_list:
        with print_lock:
            print(
                f"[{model_name}] Новости для {symbol_for_news} не найдены или не удалось загрузить. Добавляем нулевой sentiment.")
        df_out = df_candles_input.copy()
        df_out[sentiment_col_name_const] = 0.0
        return df_out

    df_news = pd.DataFrame(raw_news_list)
    df_news = df_news[['datetime', 'headline', 'summary', 'source', 'id']].copy()  # Выбираем нужные колонки
    df_news['datetime'] = pd.to_datetime(df_news['datetime'], unit='s', utc=True).dt.tz_localize(None)  # В наивный UTC
    df_news = df_news.drop_duplicates(subset=['id'])  # Удаляем дубликаты по ID новости
    df_news = df_news.sort_values(by='datetime')

    # Ограничение на количество новостей для анализа (для производительности)
    max_news_to_analyze = news_processing_config.get("max_news_per_symbol_for_period", 200)
    if len(df_news) > max_news_to_analyze:
        with print_lock:
            print(
                f"[{model_name}] Слишком много новостей ({len(df_news)}), ограничиваем до {max_news_to_analyze} самых свежих для анализа.")
        df_news = df_news.tail(max_news_to_analyze)

    # 2. Анализ Sentiment новостей
    df_news_with_sentiment = analyze_news_sentiment_bert(
        news_df=df_news,
        tokenizer=bert_tokenizer_global,
        model=bert_model_global,
        text_column='headline',
        model_name=model_name
    )

    # 3. Сопоставление новостей со свечами
    df_candles_with_news_feature = map_news_to_candles(
        df_candles=df_candles_input,
        df_news_with_sentiment=df_news_with_sentiment,
        aggregation_lookback_timedelta=news_processing_config.get("aggregation_lookback_timedelta"),
        sentiment_col_name=sentiment_col_name_const,
        model_name=model_name
    )

    return df_candles_with_news_feature



In [ ]:
def add_technical_indicators(df: pd.DataFrame, model_name="") -> pd.DataFrame:
    # ... (почти без изменений, просто убедиться, что не удаляет sentiment, если он есть) ...
    if not isinstance(df.index, pd.DatetimeIndex):
        try:
            df.index = pd.to_datetime(df.index)
        except Exception as e:
            with print_lock:
                print(f"[{model_name}] Ошибка: Индекс DataFrame не DatetimeIndex: {e}")
            return df

    is_future_ta = "-future_TA" in model_name
    if not is_future_ta:
        with print_lock:
            print(f"[{model_name}] Добавление технических индикаторов... Исходный размер df: {df.shape}")

    df_ta = df.copy()  # Работаем с копией
    # Сохраним колонку sentiment, если она есть, чтобы не потерять
    existing_sentiment_col = None
    if NEWS_SENTIMENT_COLUMN_NAME in df_ta.columns:
        existing_sentiment_col = df_ta[NEWS_SENTIMENT_COLUMN_NAME].copy()

    rename_map = {}
    for col_standard in ['open', 'high', 'low', 'close', 'volume']:
        if col_standard not in df_ta.columns and col_standard.capitalize() in df_ta.columns:
            rename_map[col_standard.capitalize()] = col_standard
    if rename_map:
        df_ta.rename(columns=rename_map, inplace=True)

    # Приводим все нужные колонки к нижнему регистру, если они есть
    ohlcv_cols_to_lower = {col: col.lower() for col in df_ta.columns if
                           col.lower() in ['open', 'high', 'low', 'close', 'volume']}
    df_ta.rename(columns=ohlcv_cols_to_lower, inplace=True)

    # Проверка наличия базовых колонок
    required_ohlcv = ['open', 'high', 'low', 'close']  # volume опционально для некоторых TA
    missing_cols = [col for col in required_ohlcv if col not in df_ta.columns]
    if missing_cols:
        with print_lock:
            print(
                f"[{model_name}] Предупреждение: Отсутствуют колонки для ТА: {missing_cols}. Некоторые ТА могут быть не рассчитаны.")
        # Если sentiment был, восстанавливаем его и возвращаем как есть
        if existing_sentiment_col is not None: df_ta[NEWS_SENTIMENT_COLUMN_NAME] = existing_sentiment_col
        return df_ta

    for period in SMA_PERIODS:
        df_ta[f'sma_{period}'] = ta.sma(df_ta['close'], length=period)
    macd = ta.macd(df_ta['close'], fast=MACD_FAST, slow=MACD_SLOW, signal=MACD_SIGNAL)
    if macd is not None and not macd.empty: df_ta = pd.concat([df_ta, macd], axis=1)
    df_ta['atr'] = ta.atr(df_ta['high'], df_ta['low'], df_ta['close'], length=ATR_PERIOD)
    bbands = ta.bbands(df_ta['close'], length=BBANDS_PERIOD, std=BBANDS_STD_DEV)
    if bbands is not None and not bbands.empty: df_ta = pd.concat([df_ta, bbands], axis=1)
    df_ta['rsi'] = ta.rsi(df_ta['close'], length=RSI_PERIOD)
    stoch = ta.stoch(df_ta['high'], df_ta['low'], df_ta['close'], k=STOCH_K, d=STOCH_D, smooth_k=STOCH_SMOOTH_K)
    if stoch is not None and not stoch.empty: df_ta = pd.concat([df_ta, stoch], axis=1)

    if 'volume' in df_ta.columns:
        df_ta['obv'] = ta.obv(df_ta['close'], df_ta['volume'])
    elif not is_future_ta:
        with print_lock:
            print(f"[{model_name}] Предупреждение: Колонка 'volume' отсутствует, OBV не рассчитан.")

    if 'volume' in df_ta.columns and 'close' in df_ta.columns and 'high' in df_ta.columns and 'low' in df_ta.columns:
        # Расчет VWAP безопаснее
        if not df_ta['volume'].empty and not df_ta['volume'].isnull().all():
            typical_price = (df_ta['high'] + df_ta['low'] + df_ta['close']) / 3
            df_ta['vwap'] = ta.vwap(df_ta['high'], df_ta['low'], df_ta['close'], df_ta['volume'])
        else:
            df_ta['vwap'] = np.nan
            if not is_future_ta:
                with print_lock:
                    print(
                        f"[{model_name}] Предупреждение: Колонка 'volume' пуста или вся из NaN, VWAP не рассчитан корректно.")
    elif not is_future_ta:
        with print_lock:
            print(f"[{model_name}] Предупреждение: Не удалось рассчитать VWAP, отсутствуют необходимые колонки.")

    df_ta.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Восстанавливаем колонку sentiment, если она была
    if existing_sentiment_col is not None:
        df_ta[NEWS_SENTIMENT_COLUMN_NAME] = existing_sentiment_col

    if not is_future_ta:
        initial_nans = df_ta.isnull().sum().max()
        with print_lock:
            print(f"[{model_name}] Технические индикаторы добавлены. NaN в начале: ~{initial_nans}")
            print(f"[{model_name}] Размер df после ТА (до dropna): {df_ta.shape}")
    return df_ta



In [ ]:
def remove_outliers_zscore(df, column_name, threshold=3, model_name=""):
    # ... (без изменений) ...
    if column_name not in df.columns:
        with print_lock: print(f"[{model_name}] Колонка '{column_name}' не найдена для удаления выбросов.")
        return df.copy()
    series = df[column_name];
    mean = series.mean();
    std = series.std()
    if std == 0 or pd.isna(std):
        with print_lock: print(f"[{model_name}] Std для '{column_name}' 0/NaN. Выбросы не удаляются.")
        return df.copy()
    z_scores = (series - mean) / std;
    outliers_mask = np.abs(z_scores) > threshold
    num_outliers = outliers_mask.sum();
    df_cleaned = df.copy()
    if num_outliers > 0:
        with print_lock:
            print(f"\n[{model_name}] Найдено/удалено {num_outliers} выбросов в '{column_name}' (Z > {threshold}).")
        df_cleaned = df[~outliers_mask]
    else:
        with print_lock:
            print(f"\n[{model_name}] Выбросы в '{column_name}' (Z > {threshold}) не найдены.")
    with print_lock:
        print(f"[{model_name}] DataFrame после удаления выбросов: {df_cleaned.shape}")
    return df_cleaned



In [ ]:
def plot_data_with_outliers(original_df, cleaned_df, column_name, threshold, model_name="", timeframe_td=None):
    # ... (без изменений) ...
    if original_df.empty or column_name not in original_df.columns:
        with print_lock: print(f"[{model_name}] Невозможно построить график выбросов: DataFrame пуст/нет колонки.")
        return

    plt.figure(figsize=(15, 7))
    ax1 = plt.subplot(2, 1, 1)
    if timeframe_td is not None:
        plotted_ts_orig, mapped_values_orig = generate_plotted_times_and_values(
            original_df.index, original_df[column_name], timeframe_td
        )
        if len(plotted_ts_orig) > 0:
            ax1.plot(plotted_ts_orig, mapped_values_orig, label=f'Исходные ({column_name})')
            mean_orig = original_df[column_name].mean()
            std_orig = original_df[column_name].std()
            if std_orig > 0 and not pd.isna(std_orig):
                outliers_mask_bool = np.abs((original_df[column_name] - mean_orig) / std_orig) > threshold
                outliers_orig_df = original_df[outliers_mask_bool]
                if not outliers_orig_df.empty:
                    map_real_to_plotted = pd.Series(data=plotted_ts_orig, index=original_df.index)
                    valid_outlier_indices = outliers_orig_df.index.intersection(map_real_to_plotted.index)
                    if not valid_outlier_indices.empty:
                        outliers_plotted_times = map_real_to_plotted.loc[valid_outlier_indices]
                        ax1.scatter(outliers_plotted_times, outliers_orig_df.loc[valid_outlier_indices, column_name],
                                    color='red', label=f'Выбросы (Z > {threshold})', s=50, zorder=5)
        else:
            ax1.plot(original_df.index, original_df[column_name], label=f'Исходные ({column_name})')
            mean_orig = original_df[column_name].mean();
            std_orig = original_df[column_name].std()
            if std_orig > 0 and not pd.isna(std_orig):
                outliers = original_df[np.abs((original_df[column_name] - mean_orig) / std_orig) > threshold]
                if not outliers.empty: ax1.scatter(outliers.index, outliers[column_name], color='red',
                                                   label=f'Выбросы (Z > {threshold})', s=50, zorder=5)
    else:
        ax1.plot(original_df.index, original_df[column_name], label=f'Исходные ({column_name})')
        mean_orig = original_df[column_name].mean();
        std_orig = original_df[column_name].std()
        if std_orig > 0 and not pd.isna(std_orig):
            outliers = original_df[np.abs((original_df[column_name] - mean_orig) / std_orig) > threshold]
            if not outliers.empty: ax1.scatter(outliers.index, outliers[column_name], color='red',
                                               label=f'Выбросы (Z > {threshold})', s=50, zorder=5)

    plt.title(f'[{model_name}] Данные до удаления выбросов ({column_name})');
    plt.ylabel('Цена');
    plt.legend();
    plt.grid(True)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'));
    ax1.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=10))

    ax2 = plt.subplot(2, 1, 2)
    if not cleaned_df.empty and column_name in cleaned_df.columns:
        if timeframe_td is not None:
            plotted_ts_cleaned, mapped_values_cleaned = generate_plotted_times_and_values(
                cleaned_df.index, cleaned_df[column_name], timeframe_td
            )
            if len(plotted_ts_cleaned) > 0:
                ax2.plot(plotted_ts_cleaned, mapped_values_cleaned, label=f'После удаления ({column_name})',
                         color='green')
            else:
                ax2.plot(cleaned_df.index, cleaned_df[column_name], label=f'После удаления ({column_name})',
                         color='green')
        else:
            ax2.plot(cleaned_df.index, cleaned_df[column_name], label=f'После удаления ({column_name})', color='green')
    else:
        plt.text(0.5, 0.5, 'Очищенные данные отсутствуют', ha='center', va='center', transform=ax2.transAxes)

    plt.title(f'[{model_name}] Данные после удаления выбросов ({column_name})');
    plt.xlabel('Дата/время');
    plt.ylabel('Цена');
    plt.legend();
    plt.grid(True)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'));
    ax2.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=10))

    plt.tight_layout();
    plt.show()



Блок 2: Процесс предобработки данных (Pipeline)

In [ ]:
def split_sequences(X_data_scaled, y_data_scaled, n_steps_in, n_steps_out, timestamps_full_data, model_name=""):
    # ... (без изменений) ...
    X, y, times_y = [], [], []
    if len(X_data_scaled) < n_steps_in + n_steps_out:
        with print_lock: print(
            f"[{model_name}] Недостаточно данных для seq: {len(X_data_scaled)} < {n_steps_in + n_steps_out}")
        return np.array(X), np.array(y), np.array(times_y)
    for i in range(len(X_data_scaled) - n_steps_in - n_steps_out + 1):
        X.append(X_data_scaled[i:(i + n_steps_in), :])
        y.append(y_data_scaled[(i + n_steps_in):(i + n_steps_in + n_steps_out)])
        times_y.append(timestamps_full_data[i + n_steps_in])
    return np.array(X), np.array(y), np.array(times_y)



In [ ]:
def preprocess_data_pipeline(
        token, figi, days, interval,
        target_feature_col, z_threshold,
        n_steps_in, n_steps_out,
        model_name="", plot_outliers_flag=True, timeframe_td_for_plotting=None,
        # --- Новые параметры для новостей ---
        news_proc_config=None,  # Словарь конфигурации новостей
        finnhub_client_glob=None,  # Глобальный клиент Finnhub
        bert_tokenizer_glob=None,  # Глобальный токенизатор BERT
        bert_model_glob=None,  # Глобальная модель BERT
        sentiment_col_name_c=NEWS_SENTIMENT_COLUMN_NAME  # Имя колонки для sentiment
):
    with print_lock:
        print(f"--- [{model_name}] Начало процесса предобработки данных ---")
    df_raw = get_tinkoff_candles(token, figi, days, interval, model_name=model_name)
    if df_raw.empty:
        with print_lock: print(f"[{model_name}] Ошибка: не удалось загрузить данные. Предобработка прервана.")
        return None, None, None, None, None, None, None

    df_cleaned_pre_ta = remove_outliers_zscore(df_raw.copy(), target_feature_col, threshold=z_threshold,
                                               model_name=model_name)
    if plot_outliers_flag:
        plot_data_with_outliers(df_raw, df_cleaned_pre_ta, target_feature_col, z_threshold,
                                model_name=model_name, timeframe_td=timeframe_td_for_plotting)

    df_with_ta = add_technical_indicators(df_cleaned_pre_ta, model_name=model_name)

    # --- ДОБАВЛЕНИЕ НОВОСТНОЙ ФИЧИ ---
    df_with_ta_and_news = df_with_ta  # По умолчанию, если новости не обрабатываются
    if news_proc_config and news_proc_config.get("enabled", False) and \
            finnhub_client_glob and bert_tokenizer_glob and bert_model_glob:
        with print_lock:
            print(f"[{model_name}] Запуск добавления новостной фичи...")
        df_with_ta_and_news = add_news_sentiment_feature(
            df_candles_input=df_with_ta,  # Передаем df уже с ТА
            news_processing_config=news_proc_config,
            finnhub_client_global=finnhub_client_glob,
            bert_tokenizer_global=bert_tokenizer_glob,
            bert_model_global=bert_model_glob,
            sentiment_col_name_const=sentiment_col_name_c,
            model_name=model_name
        )
    elif news_proc_config and news_proc_config.get("enabled", False):
        # Если включено, но ресурсы не переданы
        with print_lock:
            print(
                f"[{model_name}] Обработка новостей включена, но ресурсы (Finnhub/BERT) не предоставлены. Добавляется нулевая колонка '{sentiment_col_name_c}'.")
        df_with_ta_and_news[sentiment_col_name_c] = 0.0
    else:  # Если новости выключены в конфиге
        # Убедимся, что колонка создана с нулями, если она будет в X_feature_names по ошибке
        # Но X_feature_names будет формироваться на основе того, что есть.
        # Если новости отключены, то sentiment_col_name_c не должна попасть в X_feature_names.
        pass  # Колонка не добавляется, если новости отключены.

    df_full_features = df_with_ta_and_news.dropna().copy()  # Удаляем NaN ПОСЛЕ добавления всех фичей
    with print_lock:
        print(f"[{model_name}] DataFrame после ТА, новостей (если есть) и dropna(): {df_full_features.shape}")
        print(f"[{model_name}] Колонки в df_full_features: {df_full_features.columns.tolist()}")

    if df_full_features.empty or len(df_full_features) < n_steps_in + n_steps_out:
        with print_lock: print(
            f"[{model_name}] Недостаточно данных после обработок ({len(df_full_features)}), надо {n_steps_in + n_steps_out}.")
        return None, None, None, None, None, None, df_full_features

    y_data_unscaled = df_full_features[[target_feature_col]].values

    # Формируем X_feature_names на основе того, что есть в df_full_features
    # Исключаем колонки, которые не должны быть предикторами (например, если случайно попали)
    # Но обычно все колонки в df_full_features к этому моменту - это предикторы или таргет.
    # Таргет мы уже выделили в y_data_unscaled.
    X_feature_names = list(df_full_features.columns)

    X_data_unscaled = df_full_features[X_feature_names].values
    with print_lock:
        print(f"[{model_name}] Признаков для X (X_feature_names): {len(X_feature_names)} -> {X_feature_names}")
        print(f"[{model_name}] Форма X_data_unscaled: {X_data_unscaled.shape}")

    scaler_X = MinMaxScaler();
    X_data_scaled = scaler_X.fit_transform(X_data_unscaled)
    scaler_y = MinMaxScaler();
    y_data_scaled = scaler_y.fit_transform(y_data_unscaled)
    timestamps_for_sequences = df_full_features.index
    X_sequences, y_sequences, times_y_sequences = split_sequences(
        X_data_scaled, y_data_scaled.flatten(),
        n_steps_in, n_steps_out, timestamps_for_sequences, model_name=model_name)

    if X_sequences.shape[0] == 0:
        with print_lock: print(f"[{model_name}] Ошибка: не удалось создать обучающие последовательности.")
        return None, None, None, scaler_X, scaler_y, X_feature_names, df_full_features
    with print_lock:
        print(f"[{model_name}] Предобработка завершена. {X_sequences.shape[0]} последовательностей.")
        print(f"[{model_name}] X_seq: {X_sequences.shape}, y_seq: {y_sequences.shape}")
        print(f"--- [{model_name}] Конец процесса предобработки данных ---\n")
    return X_sequences, y_sequences, times_y_sequences, scaler_X, scaler_y, X_feature_names, df_full_features



Блок 3: Построение, обучение и оценка модели

In [ ]:
def build_lstm_model(n_steps_in, n_steps_out, n_features):
    # ... (без изменений) ...
    model = Sequential([
        LSTM(70, return_sequences=True, input_shape=(n_steps_in, n_features)), Dropout(0.3),
        LSTM(70, return_sequences=False), Dropout(0.3),
        Dense(50, activation='relu'), Dense(n_steps_out)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model



In [ ]:
def calculate_regression_metrics(y_true, y_pred):
    # ... (без изменений) ...
    if len(y_true) == 0 or len(y_pred) == 0:
        return {m: np.nan for m in ['mse', 'rmse', 'mae', 'r2', 'mse_pct', 'rmse_pct', 'mae_pct', 'r2_pct']}
    mse_raw = mean_squared_error(y_true, y_pred);
    rmse_raw = np.sqrt(mse_raw);
    mae_raw = mean_absolute_error(y_true, y_pred)
    try:
        r2_raw = r2_score(y_true, y_pred)
    except ValueError:
        r2_raw = np.nan  # Если y_true константа
    mean_y_true = np.mean(y_true)
    if np.isclose(mean_y_true, 0) or pd.isna(mean_y_true):  # Защита от деления на ноль
        mse_pct, rmse_pct, mae_pct = np.nan, np.nan, np.nan
    else:
        mae_pct = (mae_raw / abs(mean_y_true)) * 100
        rmse_pct = (rmse_raw / abs(mean_y_true)) * 100
        mse_pct = (mse_raw / (mean_y_true ** 2)) * 100 if not np.isclose(mean_y_true ** 2, 0) else np.nan  # Доп. защита
    r2_pct = r2_raw * 100 if not np.isnan(r2_raw) else np.nan
    return {'mse': mse_raw, 'rmse': rmse_raw, 'mae': mae_raw, 'r2': r2_raw,
            'mse_pct': mse_pct, 'rmse_pct': rmse_pct, 'mae_pct': mae_pct, 'r2_pct': r2_pct}



In [ ]:
def train_fold(fold_num, train_idx, val_idx, X, y_target_scaled,
               scaler_y, n_steps_in, n_steps_out, n_features, model_name="", current_n_cv_splits=N_CV_SPLITS):
    # ... (без изменений) ...
    with print_lock: print(
        f"\n--- [{model_name}] Обучение fold {fold_num + 1}/{current_n_cv_splits} (feat={n_features}) ---")
    model = build_lstm_model(n_steps_in, n_steps_out, n_features)
    X_train, y_train_target_scaled = X[train_idx], y_target_scaled[train_idx]
    X_val, y_val_target_scaled = X[val_idx], y_target_scaled[val_idx]
    history = model.fit(X_train, y_train_target_scaled, validation_data=(X_val, y_val_target_scaled),
                        epochs=50, batch_size=32, verbose=0,
                        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)])
    preds_val_target_scaled = model.predict(X_val, verbose=0)
    y_val_inv_1step = scaler_y.inverse_transform(y_val_target_scaled[:, 0].reshape(-1, 1)).flatten()
    preds_val_inv_1step = scaler_y.inverse_transform(preds_val_target_scaled[:, 0].reshape(-1, 1)).flatten()
    fold_metrics = calculate_regression_metrics(y_val_inv_1step, preds_val_inv_1step)
    with print_lock:
        print(f"[{model_name}] Результаты fold {fold_num + 1} (1-й шаг):")
        for name, val in fold_metrics.items():
            unit = " %" if name.endswith('_pct') else ""
            print(f"  {name.upper()}: {val:.4f}{unit if not pd.isna(val) else ''}")
    return {'model': model, 'metrics': fold_metrics, 'history': history, 'fold': fold_num}



In [ ]:
def cross_validate_lstm(X_sequences, y_sequences, scaler_y,
                        n_splits, n_steps_in, n_steps_out, n_features,
                        model_name="", max_workers=None):
    # ... (без изменений) ...
    effective_max_workers = 0
    if n_splits >= 2:
        effective_max_workers = min(n_splits, max_workers if max_workers is not None else (os.cpu_count() or 1) * 2)
        effective_max_workers = max(1, effective_max_workers)
    with print_lock:
        print(f"\n--- [{model_name}] CV ({n_splits} фолдов, до {effective_max_workers} потоков) ---")
    if n_splits < 2:
        with print_lock: print(f"[{model_name}] Предупреждение: n_splits = {n_splits} < 2. CV не выполнена.")
        return None, {}
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    fold_results = [];
    futures = []
    with ThreadPoolExecutor(max_workers=effective_max_workers) as executor:
        for fold_num, (train_idx, val_idx) in enumerate(kf.split(X_sequences, y_sequences)):
            futures.append(executor.submit(train_fold, fold_num, train_idx, val_idx, X_sequences, y_sequences,
                                           scaler_y, n_steps_in, n_steps_out, n_features, model_name,
                                           current_n_cv_splits=n_splits))
        for future in as_completed(futures):
            try:
                fold_results.append(future.result())
            except Exception as e:
                with print_lock:
                    print(f"[{model_name}] Ошибка в фолде CV: {e}")
                    import traceback;
                    print(traceback.format_exc())
    if not fold_results:
        with print_lock: print(f"[{model_name}] CV не дала результатов.")
        return None, {}
    fold_results.sort(key=lambda r: r['fold'])
    best_fold_result = min(fold_results, key=lambda r: r['metrics'].get('mse', float('inf')))
    with print_lock:
        print(f"\n--- [{model_name}] CV завершена ---")
        print(
            f"[{model_name}] Лучшая модель из фолда {best_fold_result['fold'] + 1} (MSE: {best_fold_result['metrics'].get('mse', float('inf')):.4f})")
        print(f"[{model_name}] Метрики лучшей модели (валид. набор, 1-й шаг):")
        for name, val in best_fold_result['metrics'].items():
            unit = " %" if name.endswith('_pct') else ""
            print(f"  {name.upper()}: {val:.4f}{unit if not pd.isna(val) else ''}")
    return best_fold_result['model'], best_fold_result['metrics']



In [ ]:
def final_test_and_plot(model, X_test, y_test_target_scaled, scaler_y,
                        times_y_test, n_steps_out, target_feature_name, model_name="",
                        timeframe_td=None):
    # ... (без изменений) ...
    with print_lock:
        print(f"\n--- [{model_name}] Финальное тестирование ---")
    if X_test.shape[0] == 0:
        with print_lock: print(f"[{model_name}] Тестовая выборка пуста. Тест не проводится.")
        return

    preds_test_target_scaled = model.predict(X_test, verbose=0)

    y_test_inv_1step = scaler_y.inverse_transform(y_test_target_scaled[:, 0].reshape(-1, 1)).flatten()
    preds_test_inv_1step = scaler_y.inverse_transform(preds_test_target_scaled[:, 0].reshape(-1, 1)).flatten()
    test_metrics_1step = calculate_regression_metrics(y_test_inv_1step, preds_test_inv_1step)
    with print_lock:
        print(f"[{model_name}] Метрики на тест. выборке (1-й шаг):")
        for name, val in test_metrics_1step.items():
            unit = " %" if name.endswith('_pct') else "";
            print(f"  {name.upper()}: {val:.4f}{unit if not pd.isna(val) else ''}")

    result_df_1step = pd.DataFrame({
        'datetime': times_y_test,
        'actual_close_1step': y_test_inv_1step,
        'predicted_close_1step': preds_test_inv_1step
    }).set_index('datetime').sort_index()

    plt.figure(figsize=(15, 6))
    ax1 = plt.gca()
    if timeframe_td is not None:
        plotted_times, _ = generate_plotted_times_and_values(result_df_1step.index,
                                                             result_df_1step['actual_close_1step'].values, timeframe_td)
        if len(plotted_times) == len(result_df_1step):
            ax1.plot(plotted_times, result_df_1step['actual_close_1step'].values,
                     label=f'Факт ({target_feature_name}, 1-й шаг)')
            ax1.plot(plotted_times, result_df_1step['predicted_close_1step'].values,
                     label=f'Прогноз ({target_feature_name}, 1-й шаг)', linestyle='--')
        else:
            ax1.plot(result_df_1step.index, result_df_1step['actual_close_1step'],
                     label=f'Факт ({target_feature_name}, 1-й шаг)')
            ax1.plot(result_df_1step.index, result_df_1step['predicted_close_1step'],
                     label=f'Прогноз ({target_feature_name}, 1-й шаг)', linestyle='--')
    else:
        ax1.plot(result_df_1step.index, result_df_1step['actual_close_1step'],
                 label=f'Факт ({target_feature_name}, 1-й шаг)')
        ax1.plot(result_df_1step.index, result_df_1step['predicted_close_1step'],
                 label=f'Прогноз ({target_feature_name}, 1-й шаг)', linestyle='--')
    plt.title(f"[{model_name}] Факт/Прогноз ({target_feature_name}, 1-й шаг) на тест. выборке");
    plt.xlabel("Дата/время");
    plt.ylabel(f"Цена ({target_feature_name})")
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'));
    ax1.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=10))
    plt.xticks(rotation=45);
    plt.legend();
    plt.grid(True);
    plt.tight_layout();
    plt.show()

    # ... (остальные графики final_test_and_plot без изменений)



Блок 4: Рекурсивное прогнозирование будущего

In [ ]:
def predict_future_recursive(
        model, initial_X_sequence_scaled, scaler_X, scaler_y,
        all_X_feature_names,  # Список имен всех фичей, которые ожидает модель
        target_feature_name, num_total_future_steps,
        n_steps_in_model, n_steps_out_model, last_known_full_data_df,
        candle_interval_timedelta, model_name="",
        # --- Новые параметры для новостей ---
        sentiment_col_name_const=NEWS_SENTIMENT_COLUMN_NAME  # Имя колонки с sentiment'ом
):
    with print_lock:
        print(f"\n[{model_name}] Рекурсивный прогноз на {num_total_future_steps} шагов...")
    all_future_preds_y_inv_flat = [];
    current_input_X_scaled = initial_X_sequence_scaled.copy()

    # history_df это копия last_known_full_data_df, которая содержит все фичи, включая sentiment, если он был
    history_df = last_known_full_data_df.copy()  # Убедимся, что это копия для модификации

    if not isinstance(history_df.index, pd.DatetimeIndex): history_df.index = pd.to_datetime(history_df.index)
    last_real_timestamp = history_df.index[-1]

    # Проверяем, есть ли колонка sentiment в признаках модели (all_X_feature_names)
    # и в history_df (должна быть, если есть в all_X_feature_names)
    news_sentiment_active = sentiment_col_name_const in all_X_feature_names and \
                            sentiment_col_name_const in history_df.columns
    if sentiment_col_name_const in all_X_feature_names and sentiment_col_name_const not in history_df.columns:
        with print_lock:
            print(
                f"[{model_name}] Внимание: '{sentiment_col_name_const}' есть в all_X_feature_names, но отсутствует в history_df для прогноза. Sentiment будет 0.")
        # Добавляем колонку с нулями, если ее нет, но она ожидается
        history_df[sentiment_col_name_const] = 0.0
        news_sentiment_active = True  # Теперь она есть

    for _ in range(int(np.ceil(num_total_future_steps / n_steps_out_model))):
        if len(all_future_preds_y_inv_flat) >= num_total_future_steps: break
        predicted_y_scaled_chunk = model.predict(current_input_X_scaled, verbose=0)
        predicted_y_inv_chunk = scaler_y.inverse_transform(predicted_y_scaled_chunk).flatten()
        num_to_add = min(len(predicted_y_inv_chunk), num_total_future_steps - len(all_future_preds_y_inv_flat))
        all_future_preds_y_inv_flat.extend(predicted_y_inv_chunk[:num_to_add])
        if len(all_future_preds_y_inv_flat) >= num_total_future_steps: break

        new_rows_for_history_df = []
        current_synth_ts_for_loop = history_df.index[-1]  # Берем последнее время из history_df

        for i in range(len(predicted_y_inv_chunk)):  # Идем по всем предсказанным шагам в чанке
            current_synth_ts_for_loop += candle_interval_timedelta
            pred_close = predicted_y_inv_chunk[i]

            # Формируем словарь для новой строки. Он должен включать OHLCV и sentiment (если активен)
            new_row_dict = {
                'open': pred_close, 'high': pred_close, 'low': pred_close, 'close': pred_close,
                'volume': history_df['volume'].iloc[-1] if 'volume' in history_df.columns else 0
            }
            if news_sentiment_active:
                new_row_dict[sentiment_col_name_const] = 0.0  # Нейтральный sentiment для будущего

            new_rows_for_history_df.append(pd.DataFrame(new_row_dict, index=[current_synth_ts_for_loop]))

        if not new_rows_for_history_df: continue

        # Добавляем новые строки в history_df. Колонки ТА в новых строках будут NaN.
        history_df = pd.concat([history_df] + new_rows_for_history_df, sort=False)

        # Пересчитываем ТА для расширенного history_df
        # Определяем, с какого индекса пересчитывать ТА, чтобы было достаточно данных
        lookback_ta = max(SMA_PERIODS[-1] if SMA_PERIODS else 0,
                          MACD_SLOW + MACD_SIGNAL if MACD_SLOW and MACD_SIGNAL else 0,
                          ATR_PERIOD if ATR_PERIOD else 0,
                          BBANDS_PERIOD if BBANDS_PERIOD else 0,
                          RSI_PERIOD if RSI_PERIOD else 0,
                          STOCH_K + STOCH_SMOOTH_K if STOCH_K and STOCH_SMOOTH_K else 0,
                          50) + 20  # Дополнительный буфер

        # Пересчитываем ТА на достаточном срезе history_df, чтобы избежать краевых эффектов
        # и затем обновляем эти ТА в основном history_df
        start_idx_recalc = max(0, len(history_df) - (
                    lookback_ta + n_steps_out_model + n_steps_in_model))  # + n_steps_in для следующего шага
        df_for_recalc_ta = history_df.iloc[start_idx_recalc:].copy()  # Копируем срез

        # В df_for_recalc_ta ТА для новых строк будут NaN, sentiment будет 0.0
        df_recalculated_ta = add_technical_indicators(df_for_recalc_ta, model_name=f"{model_name}-future_TA")

        # Обновляем history_df пересчитанными значениями ТА (и sentiment, если он был, сохранится)
        history_df.update(df_recalculated_ta)

        if history_df.empty or len(history_df) < n_steps_in_model:
            with print_lock: print(
                f"[{model_name}] Ошибка predict_future: мало данных после ТА ({len(history_df)}). Прогноз остановлен.")
            break

        # Берем последние n_steps_in_model строк из обновленного history_df для следующей итерации
        # Убедимся, что new_X_unscaled_df содержит все колонки из all_X_feature_names
        new_X_unscaled_df = history_df[all_X_feature_names].iloc[-n_steps_in_model:]

        if len(new_X_unscaled_df) < n_steps_in_model:
            with print_lock: print(
                f"[{model_name}] Ошибка predict_future: мало данных ({len(new_X_unscaled_df)}) для X_input после ТА. Стоп.")
            break

        new_X_unscaled_vals = new_X_unscaled_df.values
        if np.isnan(new_X_unscaled_vals).any():
            # Если после пересчета ТА и обновления все еще есть NaN в нужных фичах
            with print_lock:
                print(f"[{model_name}] Внимание: NaN обнаружены в X_unscaled_vals для прогноза. Попытка заполнения...")
                # print(new_X_unscaled_df[new_X_unscaled_df.isnull().any(axis=1)]) # Отладка
            temp_df_fill = pd.DataFrame(new_X_unscaled_vals, columns=all_X_feature_names, index=new_X_unscaled_df.index)
            temp_df_fill = temp_df_fill.ffill().bfill().fillna(0)  # Более агрессивное заполнение
            new_X_unscaled_vals = temp_df_fill.values
            if np.isnan(new_X_unscaled_vals).any():
                with print_lock:
                    print(
                        f"[{model_name}] КРИТИЧЕСКАЯ ОШИБКА: Не удалось заполнить все NaN в X_unscaled_vals. Прогноз остановлен.")
                break  # Критическая ошибка, если не удалось заполнить

        new_X_scaled_vals = scaler_X.transform(new_X_unscaled_vals)
        current_input_X_scaled = new_X_scaled_vals.reshape(1, n_steps_in_model, -1)

    future_preds_array = np.array(all_future_preds_y_inv_flat)
    future_times = pd.to_datetime(
        [last_real_timestamp + candle_interval_timedelta * (i + 1) for i in range(len(future_preds_array))])
    with print_lock:
        print(f"[{model_name}] Рекурсивный прогноз завершен. Предсказано {len(future_preds_array)} шагов.")
    return future_preds_array, future_times



Блок 5: Основной поток выполнения для одной модели

Блок 5: Основной поток выполнения для одной модели

In [ ]:
def run_model_pipeline(model_config,
                       # Глобальные ресурсы передаются сюда
                       finnhub_client_glob,
                       bert_tokenizer_glob,
                       bert_model_glob):
    model_name = model_config["name"]
    print_lock.acquire();
    print(f"\n\n{'=' * 30} Processing Model: {model_name} {'=' * 30}");
    print_lock.release()
    days_history = model_config["days_history"];
    candle_timeframe = model_config["candle_timeframe"]
    n_steps_in = model_config["n_steps_in"];
    n_steps_out = model_config["n_steps_out"]
    future_predictions_count = model_config["future_predictions"]

    news_cfg_for_model = model_config.get("news_processing", {"enabled": False})


    interval_td_for_plots = get_timedelta_from_candle_interval(candle_timeframe)

    X_sequences_all, y_sequences_all, times_y_all, scaler_X, scaler_y, x_feature_names, df_full_features = preprocess_data_pipeline(
        token=TINKOFF_API_TOKEN, figi=INSTRUMENT_FIGI, days=days_history, interval=candle_timeframe,
        target_feature_col=FEATURE_COLUMN, z_threshold=Z_SCORE_THRESHOLD,
        n_steps_in=n_steps_in, n_steps_out=n_steps_out, model_name=model_name,
        plot_outliers_flag=False, # Установил plot_outliers_flag=False для ускорения
        timeframe_td_for_plotting=interval_td_for_plots,
        # Передача ресурсов и конфига для новостей
        news_proc_config=news_cfg_for_model,
        finnhub_client_glob=finnhub_client_glob,
        bert_tokenizer_glob=bert_tokenizer_glob,
        bert_model_glob=bert_model_glob,
        sentiment_col_name_c=NEWS_SENTIMENT_COLUMN_NAME
    )

    if X_sequences_all is None or X_sequences_all.shape[0] == 0:
        with print_lock: print(f"[{model_name}] Не удалось подготовить данные. Пропуск модели.")
        return

    n_total_features_X = X_sequences_all.shape[2] # Это количество фичей в X_sequences_all
    # x_feature_names - это имена этих фичей

    if X_sequences_all.shape[0] < ((1 / TEST_SPLIT_SIZE) if TEST_SPLIT_SIZE > 0 else float('inf')):
        with print_lock: print(
            f"[{model_name}] Недостаточно данных ({X_sequences_all.shape[0]}) для разделения на train/test с test_size={TEST_SPLIT_SIZE}. Пропуск модели.")
        return

    X_trainval, X_test, y_trainval_target_scaled, y_test_target_scaled, times_y_trainval, times_y_test = train_test_split(
        X_sequences_all, y_sequences_all, times_y_all, test_size=TEST_SPLIT_SIZE, shuffle=False, # shuffle=False важно для временных рядов
        random_state=RANDOM_STATE)
    with print_lock:
        print(f"\n[{model_name}] Размеры выборок: Обуч/валид (X): {X_trainval.shape}, Тест (X): {X_test.shape}")

    final_model = None;
    current_n_cv_splits = N_CV_SPLITS # Используем глобальную константу
    if X_trainval.shape[0] == 0:
        with print_lock: print(f"[{model_name}] Выборка X_trainval пуста после разделения. Пропуск обучения.")
        return
    # Корректировка current_n_cv_splits если данных мало
    if X_trainval.shape[0] < current_n_cv_splits and X_trainval.shape[0] >= 2 :
        actual_cv_splits = max(2, X_trainval.shape[0] // 2 if X_trainval.shape[0] >= 4 else 2)
        if X_trainval.shape[0] < 2: actual_cv_splits = 0 # Невозможно сделать CV
        with print_lock:
            print(
                f"[{model_name}] Мало данных для {current_n_cv_splits} фолдов ({X_trainval.shape[0]}). Используется {actual_cv_splits}.")
        current_n_cv_splits = actual_cv_splits
    elif X_trainval.shape[0] < 2: # Слишком мало данных даже для 1 фолда
        current_n_cv_splits = 0
        with print_lock:
            print(f"[{model_name}] Слишком мало данных ({X_trainval.shape[0]}) для CV или обучения.")

    if current_n_cv_splits >= 2:
        best_model_cv, _ = cross_validate_lstm(X_trainval, y_trainval_target_scaled, scaler_y, current_n_cv_splits,
                                               n_steps_in, n_steps_out, n_total_features_X, model_name=model_name,
                                               max_workers=MAX_CV_WORKERS)
        final_model = best_model_cv
    elif X_trainval.shape[0] > 0 : # Обучение одной модели если CV невозможна или не нужна, но есть данные
        with print_lock:
            print(f"[{model_name}] Обучение одной модели без CV (данных для train/val: {X_trainval.shape[0]})...")
        final_model = build_lstm_model(n_steps_in, n_steps_out, n_total_features_X)

        # Создание временного валидационного набора из X_trainval, если достаточно данных
        val_data_single = None
        train_epochs = 50
        patience_es = 10
        if len(X_trainval) > max(10, (n_steps_in + n_steps_out)) * 1.1 : # Эвристика для минимального размера val
             # Пытаемся выделить 10% под валидацию, но не менее нескольких примеров
            val_split_size_internal = max(0.1, min(5 / len(X_trainval), 0.2) if len(X_trainval) > 20 else 0.1)

            temp_X_train, temp_X_val, temp_y_train, temp_y_val = train_test_split(
                X_trainval, y_trainval_target_scaled, test_size=val_split_size_internal, shuffle=False, random_state=RANDOM_STATE
            )
            if len(temp_X_val) > 0:
                val_data_single = (temp_X_val, temp_y_val)
                X_train_single, y_train_single = temp_X_train, temp_y_train
                with print_lock:
                    print(f"[{model_name}] Создан внутренний валидационный набор: {temp_X_val.shape[0]} семплов.")
            else: # Если после сплита валидационный набор пуст
                X_train_single, y_train_single = X_trainval, y_trainval_target_scaled
                with print_lock:
                    print(f"[{model_name}] Не удалось создать валидационный набор, обучаемся на всех {X_trainval.shape[0]} семплах.")
        else: # Недостаточно данных для выделения валидационного набора
            X_train_single, y_train_single = X_trainval, y_trainval_target_scaled
            with print_lock:
                print(f"[{model_name}] Недостаточно данных для внутреннего валидационного набора, обучаемся на всех {X_trainval.shape[0]} семплах.")

        final_model.fit(X_train_single, y_train_single, epochs=train_epochs, batch_size=32, verbose=0,
                        validation_data=val_data_single,
                        callbacks=[EarlyStopping(monitor='val_loss' if val_data_single else 'loss',
                                                 patience=patience_es, restore_best_weights=True)])
    else:
        with print_lock:
            print(f"[{model_name}] Нет данных для обучения (X_trainval пуст или слишком мал).")


    if final_model:
        if X_test.shape[0] > 0:
            final_test_and_plot(final_model, X_test, y_test_target_scaled, scaler_y,
                                times_y_test, n_steps_out, FEATURE_COLUMN, model_name=model_name,
                                timeframe_td=interval_td_for_plots)
        else:
            with print_lock:
                print(f"[{model_name}] Тестовая выборка пуста, финальный тест не проводился.")

        if df_full_features is not None and not df_full_features.empty and \
                len(df_full_features) >= n_steps_in and scaler_X and scaler_y:

            # Убедимся, что last_N_real_data_unscaled_X_df содержит все колонки из x_feature_names
            # И что эти колонки в правильном порядке, как ожидает scaler_X
            # x_feature_names уже должен быть корректным списком имен фичей из preprocess_data_pipeline
            last_N_real_data_unscaled_X_df = df_full_features[x_feature_names].iloc[-n_steps_in:]

            if last_N_real_data_unscaled_X_df.isnull().values.any():
                with print_lock:
                    print(f"[{model_name}] Обнаружены NaN в данных для начального прогноза. Попытка заполнения...")
                last_N_real_data_unscaled_X_df = last_N_real_data_unscaled_X_df.ffill().bfill().fillna(0) # Более агрессивное заполнение
                if last_N_real_data_unscaled_X_df.isnull().values.any():
                    with print_lock: print(
                        f"[{model_name}] Не удалось полностью заполнить NaN в начальных данных для прогноза. Прогноз может быть неточным.")

            last_N_real_data_scaled_X = scaler_X.transform(last_N_real_data_unscaled_X_df.values)
            initial_sequence_X_for_future = last_N_real_data_scaled_X.reshape(1, n_steps_in, n_total_features_X)

            future_preds_y_inv, future_times = predict_future_recursive(
                model=final_model, initial_X_sequence_scaled=initial_sequence_X_for_future,
                scaler_X=scaler_X, scaler_y=scaler_y,
                all_X_feature_names=x_feature_names, # Передаем список имен фичей
                target_feature_name=FEATURE_COLUMN,
                num_total_future_steps=future_predictions_count,
                n_steps_in_model=n_steps_in, n_steps_out_model=n_steps_out,
                last_known_full_data_df=df_full_features.copy(), # Передаем копию df_full_features
                candle_interval_timedelta=interval_td_for_plots,
                model_name=model_name,
                sentiment_col_name_const=NEWS_SENTIMENT_COLUMN_NAME # Передаем имя колонки sentiment
            )

            if future_preds_y_inv is not None and len(future_preds_y_inv) > 0:
                future_predictions_df = pd.DataFrame(
                    {'datetime': future_times, 'predicted_value': future_preds_y_inv.flatten()}).set_index('datetime')
                with print_lock:
                    print(
                        f"\n[{model_name}] Будущие прогнозные значения ({len(future_preds_y_inv)} из {future_predictions_count} запрошенных шагов):")
                    print("Первые 5 прогнозов:");
                    print(future_predictions_df.head())
                    print("\nПоследние 5 прогнозов:");
                    print(future_predictions_df.tail())

                # График: Контекст реальных данных + Прогноз на будущее
                plt.figure(figsize=(15, 7));
                ax_fut1 = plt.gca()
                context_length = min(n_steps_in * 2, len(df_full_features[FEATURE_COLUMN]))
                real_data_context_values = df_full_features[FEATURE_COLUMN].values[-context_length:]
                real_data_context_times = df_full_features.index[-context_length:]

                plotted_context_times, context_vals_for_plot = generate_plotted_times_and_values(
                    real_data_context_times, real_data_context_values, interval_td_for_plots
                )
                if len(plotted_context_times) > 0:
                    ax_fut1.plot(plotted_context_times, context_vals_for_plot, label=f'Реальные ({FEATURE_COLUMN})', marker='.')
                else:
                    ax_fut1.plot(real_data_context_times, real_data_context_values, label=f'Реальные ({FEATURE_COLUMN})', marker='.')

                if len(future_times) > 0 and len(real_data_context_times) > 0:
                    rt_numpy = real_data_context_times.to_numpy() if isinstance(real_data_context_times, pd.DatetimeIndex) else np.asarray(real_data_context_times)
                    ft_numpy = future_times.to_numpy() if isinstance(future_times, pd.DatetimeIndex) else np.asarray(future_times)
                    combined_times = np.concatenate((rt_numpy, ft_numpy))
                    combined_values = np.concatenate((real_data_context_values, future_preds_y_inv.flatten()))
                    plotted_combined_times, _ = generate_plotted_times_and_values(combined_times, combined_values, interval_td_for_plots)

                    if len(plotted_combined_times) == len(combined_times):
                        plotted_future_segment = plotted_combined_times[len(rt_numpy):]
                        ax_fut1.plot(plotted_future_segment, future_preds_y_inv.flatten(),
                                     label=f'Прогноз на {len(future_preds_y_inv)} шагов ({FEATURE_COLUMN})', marker='x', linestyle='--')
                    else:
                        ax_fut1.plot(future_times, future_preds_y_inv.flatten(),
                                     label=f'Прогноз на {len(future_preds_y_inv)} шагов ({FEATURE_COLUMN})', marker='x', linestyle='--')
                elif len(future_times) > 0:
                    plotted_ft_only, _ = generate_plotted_times_and_values(future_times, future_preds_y_inv.flatten(), interval_td_for_plots)
                    if len(plotted_ft_only) > 0:
                        ax_fut1.plot(plotted_ft_only, future_preds_y_inv.flatten(), label=f'Прогноз ({FEATURE_COLUMN})', marker='x', linestyle='--')
                    else:
                        ax_fut1.plot(future_times, future_preds_y_inv.flatten(), label=f'Прогноз ({FEATURE_COLUMN})', marker='x', linestyle='--')

                plt.title(f'[{model_name}] Контекст и Прогноз {FEATURE_COLUMN} ({candle_timeframe.name})');
                plt.xlabel('Дата/время'); plt.ylabel(f"Цена ({FEATURE_COLUMN})")
                ax_fut1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'));
                ax_fut1.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=10))
                plt.xticks(rotation=45); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

                # График: Только будущий прогноз
                plt.figure(figsize=(15, 7));
                ax_fut2 = plt.gca()
                plotted_future_only_times, _ = generate_plotted_times_and_values(
                    future_times, future_preds_y_inv.flatten(), interval_td_for_plots
                )
                if len(plotted_future_only_times) > 0:
                    ax_fut2.plot(plotted_future_only_times, future_preds_y_inv.flatten(),
                                 label=f'Прогноз на {len(future_preds_y_inv)} шагов ({FEATURE_COLUMN})', marker='x', linestyle='-', color='green')
                else:
                    ax_fut2.plot(future_times, future_preds_y_inv.flatten(),
                                 label=f'Прогноз на {len(future_preds_y_inv)} шагов ({FEATURE_COLUMN})', marker='x', linestyle='-', color='green')
                plt.title(f'[{model_name}] Только Будущий Прогноз {FEATURE_COLUMN} ({candle_timeframe.name})');
                plt.xlabel('Дата/время'); plt.ylabel(f"Цена ({FEATURE_COLUMN})")
                ax_fut2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'));
                ax_fut2.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=10))
                plt.xticks(rotation=45); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
            else:
                with print_lock:
                    print(f"[{model_name}] Рекурсивный прогноз не вернул результатов или вернул 0 шагов.")
        else:
            with print_lock:
                print(f"[{model_name}] Мало данных/нет скейлеров/нет df_full_features для рекурсивного прогноза.")
    else:
        with print_lock:
            print(f"[{model_name}] Модель не обучена. Тест/прогноз пропущены.")

    print_lock.acquire();
    print(f"\n{'=' * 30} Завершение обработки модели: {model_name} {'=' * 30}");
    print_lock.release()


In [ ]:
# ... (описание индикаторов, если нужно)

In [ ]:

if __name__ == "__main__":
    # --- Инициализация глобальных ресурсов ---
    if FINNHUB_API_KEY:
        initialize_finnhub_client(FINNHUB_API_KEY)
    else:
        print("FINNHUB_API_KEY не найден в config.py. Функционал новостей будет недоступен.")

    if BERT_SENTIMENT_MODEL_NAME:
        load_bert_resources(BERT_SENTIMENT_MODEL_NAME)
    else:
        print("BERT_SENTIMENT_MODEL_NAME не задан. Функционал анализа Sentiment новостей будет недоступен.")
    # --- Конец инициализации ---

    BASE_DAYS_HISTORY = 30
    BASE_CANDLE_TIMEFRAME = CandleInterval.CANDLE_INTERVAL_1_MIN
    BASE_N_STEPS_IN = 150
    BASE_N_STEPS_OUT = 15
    BASE_FUTURE_PREDICTIONS_COUNT = 300

    model_configs = [
        {
            "name": "Model_Hour_TF_SBER_News",  # Изменил имя для ясности
            "days_history": BASE_DAYS_HISTORY,
            "candle_timeframe": BASE_CANDLE_TIMEFRAME,
            "n_steps_in": BASE_N_STEPS_IN,
            "n_steps_out": BASE_N_STEPS_OUT,
            "future_predictions": BASE_FUTURE_PREDICTIONS_COUNT,
            "news_processing": {  # Конфигурация новостей
                "enabled": True,  # Включить обработку новостей
                "finnhub_symbol": "SBER.ME",  # Тикер Сбербанка для Finnhub
                "aggregation_lookback_timedelta": pd.Timedelta(hours=24),
                # Агрегировать новости за последние 24ч для свечи
                "news_fetch_days_offset": 7,  # Загружать новости на 7 дней раньше первой свечи
                "max_news_per_symbol_for_period": 100  # Ограничение на анализ новостей для скорости
            }
        },
        # Можно добавить другие конфигурации, например, для другой акции или с другими параметрами
        # {
        #     "name": "Model_Hour_TF_AAPL_News",
        #     "days_history": BASE_DAYS_HISTORY,
        #     "candle_timeframe": CandleInterval.CANDLE_INTERVAL_HOUR,
        #     "n_steps_in": BASE_N_STEPS_IN,
        #     "n_steps_out": BASE_N_STEPS_OUT,
        #     "future_predictions": BASE_FUTURE_PREDICTIONS_COUNT,
        #     "news_processing": {
        #         "enabled": True,
        #         "finnhub_symbol": "AAPL", # Apple
        #         "aggregation_lookback_timedelta": pd.Timedelta(hours=24),
        #         "news_fetch_days_offset": 7,
        #         "max_news_per_symbol_for_period": 100
        #     }
        # },
        {
            "name": "Model_Hour_TF_SBER_NoNews",  # Та же модель, но без новостей для сравнения
            "days_history": BASE_DAYS_HISTORY,
            "candle_timeframe": BASE_CANDLE_TIMEFRAME,
            "n_steps_in": BASE_N_STEPS_IN,
            "n_steps_out": BASE_N_STEPS_OUT,
            "future_predictions": BASE_FUTURE_PREDICTIONS_COUNT,
            "news_processing": {
                "enabled": False  # Новости выключены
            }
        }
    ]

    MIN_DESIRED_WORKERS = 12  # Уменьшил для тестирования, чтобы избежать проблем с GPU памятью
    # ... (логика определения MAX_MODEL_WORKERS без изменений) ...
    cpu_cores = os.cpu_count()
    if cpu_cores is None:
        max_cpu_based_workers = MIN_DESIRED_WORKERS
    else:
        max_cpu_based_workers = cpu_cores
    effective_workers = min(len(model_configs), max_cpu_based_workers)
    MAX_MODEL_WORKERS = min(effective_workers, MIN_DESIRED_WORKERS) if MIN_DESIRED_WORKERS > 0 else effective_workers
    MAX_MODEL_WORKERS = max(1, MAX_MODEL_WORKERS) if len(model_configs) > 0 else 0

    print(f"INFO: Запуск обучения для {len(model_configs)} моделей.")
    print(f"INFO: MAX_MODEL_WORKERS: {MAX_MODEL_WORKERS}")

    if len(model_configs) > 0 and MAX_MODEL_WORKERS > 0:
        # Если BERT_MODEL или FINNHUB_CLIENT не инициализированы, но новости включены, будет предупреждение внутри add_news_sentiment_feature
        with ThreadPoolExecutor(max_workers=MAX_MODEL_WORKERS) as executor:
            futures = [executor.submit(run_model_pipeline, config, FINNHUB_CLIENT, BERT_TOKENIZER, BERT_MODEL) for
                       config in model_configs]
            for i, future in enumerate(as_completed(futures)):
                model_name_for_error = model_configs[i]['name'] if i < len(model_configs) else 'unknown'
                try:
                    future.result()
                except Exception as e:
                    with print_lock:
                        print(
                            f"КРИТИЧЕСКАЯ ОШИБКА (поток для модели {model_name_for_error}): Обучение завершилось с исключением: {e}")
                        import traceback

                        print(traceback.format_exc())
    elif len(model_configs) == 0:
        print("INFO: Нет конфигураций моделей для обучения.")
    else:  # MAX_MODEL_WORKERS == 0
        print(f"INFO: MAX_MODEL_WORKERS ({MAX_MODEL_WORKERS}) равен 0, обучение не будет запущено.")

    with print_lock:
        print("\n--- Работа всей программы завершена ---")